# 03 — Tactical metrics

This notebook consumes the tracking CSV generated by `02_tracking_pipeline.ipynb` and reproduces the tactical/physical analysis layer developed in the final project.

The project outputs include distance covered, average player positions, individual heatmaps, Zone Control, Team Surface and the Defensive Coverage Index (ICD).

> These are research-prototype metrics. Their interpretation depends directly on tracking quality and pitch-coordinate accuracy.


## Expected input and output

Expected input: `outputs/tracking_output.csv`

Generated plots and animations: `outputs/metrics/`


## Core analysis and visualisation functions


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, Rectangle, Arc
from matplotlib import animation
from matplotlib.animation import FuncAnimation, PillowWriter
from typing import Tuple, Dict
from importlib import reload
from pathlib import Path
from IPython.display import display, Image

FIELD_LENGTH = 105  # metros
FIELD_WIDTH = 68    # metros
ZONE_ROWS = 4       # dividir campo em x zonas longitudinais
ZONE_COLS = 4       # dividir campo em x zonas transversais

def get_teams_ids(df):
    df = df.copy()
    teamA_ids = df['jogador'].groupby(df['equipa']).unique()['A']
    teamB_ids = df['jogador'].groupby(df['equipa']).unique()['B']
    return teamA_ids, teamB_ids

def scale_coordinates(df):
    df = df.copy()
    df["x_m"] = df["x"] * FIELD_LENGTH / 100
    df["y_m"] = df["y"] * FIELD_WIDTH / 100
    return df

def draw_pitch(ax):

    ax.set_xlim(0, FIELD_LENGTH)
    ax.set_ylim(0, FIELD_WIDTH)
    ax.axis("off")


    # Linhas do campo
    lw = 1.5
    ax.add_patch(Rectangle((0, 0), FIELD_LENGTH, FIELD_WIDTH, fill=False, color="black", lw=lw))


    # Linha de meio campo
    ax.plot([FIELD_LENGTH/2, FIELD_LENGTH/2], [0, FIELD_WIDTH], color="black", lw=lw)
    ax.add_patch(Circle((FIELD_LENGTH/2, FIELD_WIDTH/2), 9.15, fill=False, color="black", lw=lw))


    # Balizas
    ax.add_patch(Rectangle((-1, (FIELD_WIDTH-7.32)/2), 1, 7.32, fill=True, color="black"))
    ax.add_patch(Rectangle((FIELD_LENGTH, (FIELD_WIDTH-7.32)/2), 1, 7.32, fill=True, color="black"))


    # Grandes áreas
    ax.add_patch(Rectangle((0, (FIELD_WIDTH-40.32)/2), 16.5, 40.32, fill=False, color="black", lw=lw))
    ax.add_patch(Rectangle((FIELD_LENGTH-16.5, (FIELD_WIDTH-40.32)/2), 16.5, 40.32, fill=False, color="black", lw=lw))


    # Pequenas áreas
    ax.add_patch(Rectangle((0, (FIELD_WIDTH-18.32)/2), 5.5, 18.32, fill=False, color="black", lw=lw))
    ax.add_patch(Rectangle((FIELD_LENGTH-5.5, (FIELD_WIDTH-18.32)/2), 5.5, 18.32, fill=False, color="black", lw=lw))


    return ax


# Posições médias

def plot_average_positions(df, out_file="posicoes_medias.png"):
    df = scale_coordinates(df)
    teamA_ids, teamB_ids = get_teams_ids(df)

    fig, ax = plt.subplots(figsize=(10, 7))
    draw_pitch(ax)

    for pid in df["jogador"].unique():
        player_df = df[df["jogador"] == pid]
        mean_x, mean_y = player_df["x_m"].mean(), player_df["y_m"].mean()
        color = "red" if pid in teamA_ids else "blue" if pid in teamB_ids else "lightgrey"
        ax.scatter(mean_x, mean_y, color=color, s=80, edgecolor="black", zorder=3)
        ax.text(mean_x+0.5, mean_y, str(int(pid)), color="black", fontsize=9, weight="bold")

    ax.set_title("Posições médias dos jogadores")
    fig.savefig(out_file, dpi=300, bbox_inches="tight", facecolor='white')
    plt.show()


# Distância percorrida

def plot_distance(df, out_file="distancia_percorrida.png"):
    teamA_ids, teamB_ids = get_teams_ids(df)
    df = scale_coordinates(df)
    distances = (
        df.sort_values(["jogador", "frame"])
          .groupby("jogador")
          .apply(lambda g: np.sum(np.sqrt(np.diff(g["x_m"])**2 + np.diff(g["y_m"])**2)))
    )
    distances_km = distances / 1000

    distances_km_sorted = distances_km.sort_values()

    colors = ["red" if pid in teamA_ids else "blue" if pid in teamB_ids else "lightgrey" for pid in distances_km.keys()]

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.barh(distances_km_sorted.index.astype(str), distances_km_sorted.values,
                   color=colors, edgecolor="black")
    ax.set_xlabel("Distância percorrida (km)")
    ax.set_title("Distância percorrida por jogador")

    for bar, value in zip(bars, distances_km_sorted.values):
        ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                f"{value:.2f}", va='center')

    fig.savefig(out_file, dpi=300, bbox_inches="tight", facecolor='white')
    plt.tight_layout()
    return fig


# Heatmap

def plot_heatmap(df, player_id, use_bins=False, out_file=None):
    df = scale_coordinates(df)
    player_df = df[df["jogador"] == player_id]

    fig, ax = plt.subplots(figsize=(10, 7))
    draw_pitch(ax)

    if use_bins:
        hb = ax.hexbin(player_df["x_m"], player_df["y_m"], gridsize=25, cmap="Reds", alpha=0.7)
        fig.colorbar(hb, ax=ax, label="Frequência")
    else:
        sns.kdeplot(x=player_df["x_m"], y=player_df["y_m"], fill=True, cmap="Reds", alpha=0.6, ax=ax, levels=50, thresh=0, bw_adjust=0.5)

    ax.set_title(f"Heatmap do Jogador {int(player_id)}")

    
    if out_file is None:
        out_file = f"heatmap_jogador_{int(player_id)}.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight", facecolor='white')
    plt.show()
    return fig


# Zone Control (estático)

def plot_zone_control(df, n_rows=ZONE_ROWS, n_cols=ZONE_COLS, out_file="zone_control.png"):
    df = scale_coordinates(df)
    teamA_ids, teamB_ids = get_teams_ids(df)

    zone_w = FIELD_LENGTH / n_cols
    zone_h = FIELD_WIDTH / n_rows
    df["zone"] = df.apply(lambda r: (int(r["x_m"]//zone_w), int(r["y_m"]//zone_h)), axis=1)


    zone_control = df.groupby(["zone", "equipa"]).size().reset_index(name="count")

    zone_map = {}
    for zone, g in zone_control.groupby("zone"):
        if len(g) > 1 and g.iloc[0]["count"] == g.iloc[1]["count"]:
            zone_map[zone] = None   # empate → zona neutra
        else:
            zone_map[zone] = g.loc[g["count"].idxmax()]["equipa"]

    fig, ax = plt.subplots(figsize=(10, 7))
    draw_pitch(ax)

    for row in range(n_rows):
        for col in range(n_cols):
            team = zone_map.get((col, row), None)
            color = "red" if team == "A" else "blue" if team == "B" else "lightgrey"
            rect = Rectangle((col*zone_w, row*zone_h), zone_w, zone_h,
                             facecolor=color, alpha=0.3, edgecolor="white")
            ax.add_patch(rect)

    for pid in df["jogador"].unique():
        player_df = df[df["jogador"] == pid]
        mean_x, mean_y = player_df["x_m"].mean(), player_df["y_m"].mean()
        color = "red" if pid in teamA_ids else "blue" if pid in teamB_ids else "lightgrey"
        ax.scatter(mean_x, mean_y, color=color, s=80, edgecolor="black", zorder=3)
        ax.text(mean_x+0.5, mean_y, str(int(pid)), color="black", fontsize=9, weight="bold")

    ax.set_title("Zone Control (estático)")
    fig.savefig(out_file, dpi=300, bbox_inches="tight", facecolor='white')
    return fig


# Zone Control (animado)

def animate_zone_control(df, n_rows=ZONE_ROWS, n_cols=ZONE_COLS, out_file="zone_control.gif"):
    df = scale_coordinates(df)

    zone_w = FIELD_LENGTH / n_cols
    zone_h = FIELD_WIDTH / n_rows
    df["zone"] = df.apply(lambda r: (int(r["x_m"]//zone_w), int(r["y_m"]//zone_h)), axis=1)

    zone_control = df.groupby(["frame", "zone", "equipa"]).size().reset_index(name="count")


    frame_zone_map = {}
    for (frame, zone), g in zone_control.groupby(["frame", "zone"]):
        if len(g) > 1 and g.iloc[0]["count"] == g.iloc[1]["count"]:
            team = None   # empate
        else:
            team = g.loc[g["count"].idxmax()]["equipa"]

        frame_zone_map.setdefault(frame, {})[zone] = team

    fig, ax = plt.subplots(figsize=(10, 7))

    def update(frame):
        ax.clear()
        draw_pitch(ax)
        ax.set_title(f"Zone Control - Frame {frame}")
        zones = frame_zone_map.get(frame, {})
        for col in range(n_cols):
            for row in range(n_rows):
                team = zones.get((col, row), None)
                color = "red" if team == "A" else "blue" if team == "B" else "lightgrey"
                rect = Rectangle((col*zone_w, row*zone_h), zone_w, zone_h,
                                 facecolor=color, alpha=0.3, edgecolor="white")
                ax.add_patch(rect)

        # Adicionar jogadores nesse frame
        players = df[df["frame"] == frame]
        for _, r in players.iterrows():
            color = "red" if r["equipa"] == "A" else "blue" if r["equipa"] == "B" else "lightgrey"
            ax.scatter(r["x_m"], r["y_m"], color=color, s=50, edgecolor="black", zorder=3)
            ax.text(r["x_m"]+0.5, r["y_m"], str(int(r["jogador"])), fontsize=8, weight="bold")

    frames_sorted = sorted(frame_zone_map.keys())[::10]

    ani = animation.FuncAnimation(fig, update, frames=frames_sorted, interval=1000)
    ani.save(out_file, writer=PillowWriter(fps=1))
    plt.close(fig)


## Collective metrics

Additional functions used for Team Surface, ICD and combined average-position heatmaps.


In [ ]:
"""
Métricas e gráficos (sem velocidades):
1) Team Surface (fecho convexo) por equipa (eixo do tempo em segundos)
2) ICD (Índice de Cobertura Defensiva) por equipa (tempo em segundos)
3) Heatmap por POSIÇÕES MÉDIAS (por equipa e combinado A+B)
4) Animação do heatmap combinado (títulos com tempo em s)
5) Gráficos: séries de Team Surface e de ICD (x = segundos)
"""

from typing import Tuple, Dict
import numpy as np
import pandas as pd

FIELD_LENGTH = 105.0
FIELD_WIDTH  = 68.0
FIGSIZE = (8, 5.5)


def scale_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "x" not in out.columns or "y" not in out.columns:
        out["x"] = out["coord_x"] * 100.0
        out["y"] = 100.0 - (out["coord_y"] * 100.0)
    out["x_m"] = out["x"] * FIELD_LENGTH / 100.0
    out["y_m"] = out["y"] * FIELD_WIDTH  / 100.0
    return out

def estimate_frame_dt(df: pd.DataFrame) -> float:
    """Δt médio entre frames (s). Usa 'time_s' se existir; senão 0.04s (~25 fps)."""
    if "time_s" in df.columns:
        dts = (df.groupby("frame")["time_s"].first().sort_index().diff()).dropna()
        dt = float(dts.median()) if not dts.empty else 0.04
        if dt <= 0 or not np.isfinite(dt): dt = 0.04
        return dt
    return 0.04

def frame_to_seconds_map(df: pd.DataFrame) -> Dict[int, float]:
    """Devolve um dicionário frame -> segundos (0 no primeiro frame)."""
    if "time_s" in df.columns:
        series = df.groupby("frame")["time_s"].first().sort_index()
        t0 = float(series.iloc[0])
        return {int(f): float(t - t0) for f, t in series.items()}
    dt = estimate_frame_dt(df)
    f0 = int(df["frame"].min())
    return {int(f): (int(f) - f0) * dt for f in sorted(df["frame"].unique())}

def draw_pitch(ax):
    import matplotlib.patches as mpatch
    lw=1.5
    ax.set_xlim(0, FIELD_LENGTH); ax.set_ylim(0, FIELD_WIDTH)
    ax.axis("off")
    ax.add_patch(mpatch.Rectangle((0,0), FIELD_LENGTH, FIELD_WIDTH, fill=False, lw=lw, ec="black"))
    ax.plot([FIELD_LENGTH/2, FIELD_LENGTH/2], [0, FIELD_WIDTH], lw=lw, c="black")
    ax.add_patch(mpatch.Circle((FIELD_LENGTH/2, FIELD_WIDTH/2), 9.15, fill=False, lw=lw, ec="black"))
    ax.add_patch(mpatch.Rectangle((-1,(FIELD_WIDTH-7.32)/2), 1,7.32, fc="black"))
    ax.add_patch(mpatch.Rectangle((FIELD_LENGTH,(FIELD_WIDTH-7.32)/2), 1,7.32, fc="black"))
    ax.add_patch(mpatch.Rectangle((0,(FIELD_WIDTH-40.32)/2), 16.5,40.32, fill=False, lw=lw, ec="black"))
    ax.add_patch(mpatch.Rectangle((FIELD_LENGTH-16.5,(FIELD_WIDTH-40.32)/2), 16.5,40.32, fill=False, lw=lw, ec="black"))
    ax.add_patch(mpatch.Rectangle((0,(FIELD_WIDTH-18.32)/2), 5.5,18.32, fill=False, lw=lw, ec="black"))
    ax.add_patch(mpatch.Rectangle((FIELD_LENGTH-5.5,(FIELD_WIDTH-18.32)/2), 5.5,18.32, fill=False, lw=lw, ec="black"))

# --------------------- Team Surface ---------------------
def _monotone_chain_convex_hull(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    if len(points) <= 1: return points.copy()
    pts = points[np.lexsort((points[:,1], points[:,0]))]
    def cross(o,a,b): return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lower=[]
    for p in pts:
        while len(lower)>=2 and cross(lower[-2], lower[-1], p) <= 0: lower.pop()
        lower.append(tuple(p))
    upper=[]
    for p in reversed(pts):
        while len(upper)>=2 and cross(upper[-2], upper[-1], p) <= 0: upper.pop()
        upper.append(tuple(p))
    return np.array(lower[:-1]+upper[:-1], dtype=float)

def _polygon_area(points: np.ndarray) -> float:
    if points is None or len(points)<3: return 0.0
    x=points[:,0]; y=points[:,1]
    return 0.5*abs(np.dot(x, np.roll(y,-1)) - np.dot(y, np.roll(x,-1)))

def team_surface_timeseries(df: pd.DataFrame) -> pd.DataFrame:
    dfa = scale_coordinates(df).copy()
    rows=[]
    for (frame, team), g in dfa.groupby(["frame","equipa"]):
        hull = _monotone_chain_convex_hull(g[["x_m","y_m"]].to_numpy())
        area = _polygon_area(hull)
        rows.append((frame, team, area))
    return pd.DataFrame(rows, columns=["frame","equipa","area_m2"])\
             .sort_values(["equipa","frame"])

def summarize_team_surface(ts: pd.DataFrame) -> pd.DataFrame:
    return (ts.groupby("equipa")["area_m2"]
              .agg(media_area_m2="mean", mediana_area_m2="median",
                   max_area_m2="max", min_area_m2="min").reset_index())

# --------------------- ICD ---------------------
def defensive_coverage_index(df: pd.DataFrame, radius: float=8.0, grid: float=1.0) -> pd.DataFrame:
    """ICD por equipa e frame (grelha). ICD = 1 - (Área_unida / (N * π r²))"""
    dfa = scale_coordinates(df).copy()
    xs = np.arange(0+grid/2, FIELD_LENGTH, grid)
    ys = np.arange(0+grid/2, FIELD_WIDTH,  grid)
    X,Y = np.meshgrid(xs, ys)
    GP = np.stack([X.ravel(), Y.ravel()], axis=1)
    r2 = radius*radius; cell_area = grid*grid
    rows=[]
    for (frame, team), g in dfa.groupby(["frame","equipa"]):
        pts = g[["x_m","y_m"]].to_numpy()
        if len(pts)==0: continue
        covered = np.zeros(len(GP), dtype=bool)
        for p in pts:
            dx = GP[:,0]-p[0]; dy = GP[:,1]-p[1]
            covered |= (dx*dx+dy*dy) <= r2
        union_area = covered.sum()*cell_area
        sum_areas = len(pts) * (np.pi * r2)
        icd = 1.0 - (union_area / sum_areas) if sum_areas>0 else np.nan
        rows.append((frame, team, icd, union_area, sum_areas))
    return pd.DataFrame(rows, columns=["frame","equipa","ICD","area_unida","soma_areas"])\
             .sort_values(["equipa","frame"])

def summarize_icd(icd_df: pd.DataFrame) -> pd.DataFrame:
    return (icd_df.groupby("equipa")["ICD"]
              .agg(media_ICD="mean", mediana_ICD="median",
                   max_ICD="max", min_ICD="min").reset_index())

# --------------------- Heatmaps por posições médias ---------------------
def _mean_positions(df: pd.DataFrame, team: str) -> np.ndarray:
    dfa = scale_coordinates(df)
    return (dfa[dfa['equipa']==team].groupby('jogador')[['x_m','y_m']].mean().to_numpy())

def team_average_position_heatmap(
    df: pd.DataFrame, team: str='A', method: str='gaussian',
    bins: Tuple[int,int]=(120,78), radius: float=8.0, sigma: float=6.0,
    out_file: str='heatmap_team_avg.png'
) -> str:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatch

    pts = _mean_positions(df, team)
    x_edges = np.linspace(0, FIELD_LENGTH, bins[0]+1)
    y_edges = np.linspace(0, FIELD_WIDTH,  bins[1]+1)
    Xc = (x_edges[:-1]+x_edges[1:])/2
    Yc = (y_edges[:-1]+y_edges[1:])/2
    X,Y = np.meshgrid(Xc, Yc)
    H = np.zeros_like(X, dtype=float)

    if method=='uniform':
        r2 = radius*radius
        for (px,py) in pts: H += ((X-px)**2 + (Y-py)**2 <= r2).astype(float)
    else:
        for (px,py) in pts: H += np.exp(-(((X-px)**2 + (Y-py)**2)/(2*sigma*sigma)))

    if H.max()>0: H = H/H.max()

    fig, ax = plt.subplots(figsize=FIGSIZE)
    draw_pitch(ax)
    ax.imshow(H, extent=[0, FIELD_LENGTH, 0, FIELD_WIDTH], origin='lower',
              cmap=('Reds' if team=='A' else 'Blues'), alpha=0.75, aspect='auto')

    face = 'red' if team=='A' else 'blue'
    for (px,py) in pts: ax.scatter(px, py, s=46, edgecolor='black', facecolor=face, zorder=3)

    ax.legend(handles=[mpatch.Patch(color=face, label=f'Equipa {team}')],
              loc='upper right', frameon=True)
    ax.set_title(f'Heatmap (posições médias) – Equipa {team} – {method}')
    fig.savefig(out_file, dpi=200, bbox_inches='tight'); plt.close(fig)
    return out_file

def team_average_position_heatmap_both(
    df: pd.DataFrame, method: str='gaussian',
    bins: Tuple[int,int]=(120,78), radius: float=8.0, sigma: float=6.0,
    teamA_attack: str='right', out_file: str='heatmap_avg_both.png'
) -> str:
    """Heatmap combinado A (Reds) + B (Blues) com jogadores pintados e seta única da direção de A."""
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatch

    ptsA = _mean_positions(df, 'A'); ptsB = _mean_positions(df, 'B')

    x_edges = np.linspace(0, FIELD_LENGTH, bins[0]+1)
    y_edges = np.linspace(0, FIELD_WIDTH,  bins[1]+1)
    Xc = (x_edges[:-1]+x_edges[1:])/2; Yc = (y_edges[:-1]+y_edges[1:])/2
    X,Y = np.meshgrid(Xc, Yc)
    HA = np.zeros_like(X, dtype=float); HB = np.zeros_like(X, dtype=float)

    if method=='uniform':
        r2 = radius*radius
        for (px,py) in ptsA: HA += ((X-px)**2 + (Y-py)**2 <= r2).astype(float)
        for (px,py) in ptsB: HB += ((X-px)**2 + (Y-py)**2 <= r2).astype(float)
    else:
        for (px,py) in ptsA: HA += np.exp(-(((X-px)**2 + (Y-py)**2)/(2*sigma*sigma)))
        for (px,py) in ptsB: HB += np.exp(-(((X-px)**2 + (Y-py)**2)/(2*sigma*sigma)))

    if HA.max()>0: HA = HA/HA.max()
    if HB.max()>0: HB = HB/HB.max()

    fig, ax = plt.subplots(figsize=FIGSIZE)
    draw_pitch(ax)
    ax.imshow(HA, extent=[0, FIELD_LENGTH, 0, FIELD_WIDTH], origin='lower',
              cmap='Reds',  alpha=0.65, aspect='auto')
    ax.imshow(HB, extent=[0, FIELD_LENGTH, 0, FIELD_WIDTH], origin='lower',
              cmap='Blues', alpha=0.65, aspect='auto')

    for (px,py) in ptsA: ax.scatter(px, py, s=48, edgecolor='black', facecolor='red',  zorder=3)
    for (px,py) in ptsB: ax.scatter(px, py, s=48, edgecolor='black', facecolor='blue', zorder=3)

    ax.legend(handles=[mpatch.Patch(color='red', label='Equipa A'),
                       mpatch.Patch(color='blue', label='Equipa B')],
              loc='upper right', frameon=True)

    y_arrow = FIELD_WIDTH - 3
    if teamA_attack.lower() in ('right','direita','ltr'):
        ax.annotate('A ataca', xy=(FIELD_LENGTH*0.10, y_arrow), xytext=(FIELD_LENGTH*0.35, y_arrow),
                    arrowprops=dict(arrowstyle='->', lw=2, color='red'),
                    color='red', fontsize=11, ha='left', va='center')
    else:
        ax.annotate('A ataca', xy=(FIELD_LENGTH*0.90, y_arrow), xytext=(FIELD_LENGTH*0.65, y_arrow),
                    arrowprops=dict(arrowstyle='->', lw=2, color='red'),
                    color='red', fontsize=11, ha='right', va='center')

    ax.set_title('Heatmap (posições médias) – A (Reds) vs B (Blues)')
    fig.savefig(out_file, dpi=200, bbox_inches='tight'); plt.close(fig)
    return out_file

# --------------------- Animação do heatmap combinado ---------------------
def animate_heatmap_combined(
    df: pd.DataFrame, method: str='gaussian',
    bins: Tuple[int,int]=(80,52), radius: float=8.0, sigma: float=6.0,
    cumulative: bool=False, window_seconds: float=5.0,
    fps: int=6, step: int=3,
    teamA_attack: str='right',
    out_file: str='heatmap_combined.gif'
) -> str:
    import matplotlib.pyplot as plt
    from matplotlib import animation
    from matplotlib.animation import PillowWriter

    dfa = scale_coordinates(df).sort_values("frame").copy()
    frames_all = sorted(dfa["frame"].unique())[::max(1,step)]
    tmap = frame_to_seconds_map(dfa)

    if not cumulative:
        dt = estimate_frame_dt(dfa)
        win_frames = max(1, int(round(window_seconds / max(dt, 1e-6))))

    x_edges = np.linspace(0, FIELD_LENGTH, bins[0]+1)
    y_edges = np.linspace(0, FIELD_WIDTH,  bins[1]+1)
    Xc = (x_edges[:-1]+x_edges[1:])/2; Yc = (y_edges[:-1]+y_edges[1:])/2
    X,Y = np.meshgrid(Xc, Yc)

    def density(points: np.ndarray, is_gauss=True):
        H = np.zeros_like(X, dtype=float)
        if len(points)==0: return H
        if is_gauss:
            for (px,py) in points: H += np.exp(-(((X-px)**2 + (Y-py)**2)/(2*sigma*sigma)))
        else:
            r2 = radius*radius
            for (px,py) in points: H += ((X-px)**2 + (Y-py)**2 <= r2).astype(float)
        if H.max()>0: H = H/H.max()
        return H

    fig, ax = plt.subplots(figsize=FIGSIZE)

    def draw_one(frame):
        ax.clear(); draw_pitch(ax)

        if cumulative:
            sub = dfa[dfa["frame"] <= frame]
        else:
            start = max(frames_all[0], frame - win_frames)
            sub = dfa[(dfa["frame"] >= start) & (dfa["frame"] <= frame)]

        Apts = sub[sub["equipa"]=='A'][["x_m","y_m"]].to_numpy()
        Bpts = sub[sub["equipa"]=='B'][["x_m","y_m"]].to_numpy()

        HA = density(Apts, is_gauss=(method!='uniform'))
        HB = density(Bpts, is_gauss=(method!='uniform'))

        ax.imshow(HA, extent=[0, FIELD_LENGTH, 0, FIELD_WIDTH], origin='lower',
                  cmap='Reds', alpha=0.65, aspect='auto')
        ax.imshow(HB, extent=[0, FIELD_LENGTH, 0, FIELD_WIDTH], origin='lower',
                  cmap='Blues', alpha=0.65, aspect='auto')

        now = dfa[dfa["frame"]==frame]
        A_now = now[now["equipa"]=='A']; B_now = now[now["equipa"]=='B']
        ax.scatter(A_now["x_m"], A_now["y_m"], s=32, edgecolor='black', facecolor='red', zorder=3)
        ax.scatter(B_now["x_m"], B_now["y_m"], s=32, edgecolor='black', facecolor='blue', zorder=3)

        # título em segundos
        t = tmap.get(int(frame), 0.0)
        ax.set_title(f"Heatmap combinado – t = {t:.1f}s – "
                     f"{'cumulativo' if cumulative else f'janela {window_seconds:.1f}s'}")

        import matplotlib.patches as mpatch
        ax.legend(handles=[mpatch.Patch(color='red', label='Equipa A'),
                           mpatch.Patch(color='blue', label='Equipa B')],
                  loc='upper right', frameon=True)

    ani = animation.FuncAnimation(fig, draw_one, frames=frames_all, interval=max(1,1000//fps))
    ani.save(out_file, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return out_file

# --------------------- Gráficos (eixo X em segundos) ---------------------
def plot_team_surface_timeseries(ts: pd.DataFrame, df_source: pd.DataFrame,
                                 out_file: str='plot_team_surface.png') -> str:
    """Plota Team Surface com eixo X em segundos (mapeado a partir de df_source)."""
    import matplotlib.pyplot as plt
    tmap = frame_to_seconds_map(df_source)
    ts2 = ts.copy()
    ts2["t_s"] = ts2["frame"].map(tmap)
    plt.figure(figsize=FIGSIZE)
    piv = ts2.pivot(index="t_s", columns="equipa", values="area_m2")
    piv.plot(ax=plt.gca())
    plt.xlabel("Tempo (s)"); plt.ylabel("Área (m²)")
    plt.title("Team Surface ao longo do tempo")
    plt.tight_layout(); plt.savefig(out_file, dpi=200, bbox_inches="tight"); plt.close()
    return out_file

def plot_icd_timeseries(icd: pd.DataFrame, df_source: pd.DataFrame,
                        out_file: str='plot_icd.png') -> str:
    """Plota ICD com eixo X em segundos (mapeado a partir de df_source)."""
    import matplotlib.pyplot as plt
    tmap = frame_to_seconds_map(df_source)
    icd2 = icd.copy()
    icd2["t_s"] = icd2["frame"].map(tmap)
    plt.figure(figsize=FIGSIZE)
    piv = icd2.pivot(index="t_s", columns="equipa", values="ICD")
    piv.plot(ax=plt.gca())
    plt.xlabel("Tempo (s)"); plt.ylabel("ICD")
    plt.title("Índice de Cobertura Defensiva (ICD)")
    plt.tight_layout(); plt.savefig(out_file, dpi=200, bbox_inches="tight"); plt.close()
    return out_file


## Load tracking data


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_CSV = PROJECT_ROOT / "outputs" / "tracking_output.csv"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_CSV)

# Normalised coordinates used by the original visualisation functions.
df['x'] = df['coord_x'] * 100
df['y'] = 100 - (df['coord_y'] * 100)


## Core visualisations


In [ ]:
average_positions = plot_average_positions(
    df, out_file=str(METRICS_DIR / "average_positions.png")
)

distance = plot_distance(
    df, out_file=str(METRICS_DIR / "distance_covered.png")
)

heatmap_1 = plot_heatmap(
    df, 1, out_file=str(METRICS_DIR / "player_1_heatmap.png")
)

zone_control = plot_zone_control(
    df, out_file=str(METRICS_DIR / "zone_control.png")
)

animate_zone_control(
    df, out_file=str(METRICS_DIR / "zone_control.gif")
)


## Team Surface, ICD and combined heatmaps

The following section reproduces the collective metrics and summary visualisations used in the academic project.


In [ ]:
# 1) Team Surface
ts = team_surface_timeseries(df)
display(summarize_team_surface(ts))
ts_img = plot_team_surface_timeseries(ts, df_source=df, out_file=str(METRICS_DIR / 'team_surface.png'))

# 2) ICD
icd = defensive_coverage_index(df, radius=8.0, grid=1.0)
display(summarize_icd(icd))
icd_img = plot_icd_timeseries(icd, df_source=df, out_file=str(METRICS_DIR / 'defensive_coverage_index.png'))

# 3) Heatmaps por posições médias
hA   = team_average_position_heatmap(df, team='A', method='gaussian', sigma=6.0,
                                     out_file=str(METRICS_DIR / 'heatmap_avg_teamA.png'))
hB   = team_average_position_heatmap(df, team='B', method='gaussian', sigma=6.0,
                                     out_file=str(METRICS_DIR / 'heatmap_avg_teamB.png'))
hBoth = team_average_position_heatmap_both(df, method='gaussian', sigma=6.0,
                                           teamA_attack='right',
                                           out_file=str(METRICS_DIR / 'heatmap_avg_both.png'))

# 4) Animação do heatmap combinado
gif_path = animate_heatmap_combined(df, method='gaussian', sigma=6.0,
                                    cumulative=False, window_seconds=5,
                                    fps=6, step=3, teamA_attack='right',
                                    out_file=str(METRICS_DIR / 'heatmap_combined.gif'))

# Mostrar tudo no output
for p in [ts_img, icd_img, hA, hB, hBoth]:
    display(Image(filename=p))
display(Image(filename=gif_path))

print("✅ Guardado:", [ts_img, icd_img, hA, hB, hBoth, gif_path])


## Interpretation note

Derived metrics should not be treated as ground-truth physical or tactical measurements. Broadcast camera motion, partial pitch visibility, occlusion, identity instability and coordinate-estimation error propagate into downstream outputs. See `docs/limitations.md`.
